### <span style="color: #4FC1FF;">O que é um Reducer?</span>
Imagine que temos uma variavel historico que guarda mensagens de um chatbot. Inicialmente:

In [7]:
historico = ['Olá']

Depois o agente produz uma nova mensagem:

In [8]:
nova_mensagem = ['como posso ajudar voce?']

#### <span style="color: #4FC1FF;">Como fariamos para atualizar o historico ?</span>

Uma possibilidade seria sobrescrever, fazendo:
historico = nova_mensagem

mas assim perderiamos a primeira mensagem, outra solução é concatenar. Fazendo:

In [9]:
historico = historico + nova_mensagem
print(historico)

['Olá', 'como posso ajudar voce?']


#### <span style="color: #4FC1FF;">No LangGraph os reducers determinam como as alterações feitas pelos nodes seram incorporadas no estado </span>

A Seguir alguns exemplos de reducers personalizados:

#### <span style="color: #4FC1FF;">[1] Os Unicos</span>
O objetivo dessa função é juntar duas listas sem adicionar elementos repetidos

In [10]:
def unicos(atual : list, novo :list)-> list:
    return list(dict.fromkeys((atual or [])+(novo or [])))

##### Como ela funciona ?
[1] (atual or []) + (novo or []) : Se atual estiver vazio ou for None utilizamos uma lista vazia, o mesmo vale para novo.

[2] +: concatenamos as duas listas.

[3] dict.fromKeys : cria um dicionário utilizando os elementos da lista como chaves.
    > Os dicts não permitem elementos repetidos
    
[4] list() : converte de volta para lista

In [11]:
atual = ["Python", "Java"]
novo = ["Java", "C++"]

resultado = unicos(atual,novo)
print(resultado)

['Python', 'Java', 'C++']


#### <span style="color: #4FC1FF;">[2] Mesclar</span>
O objetivo dessa função é juntar dois dicts

In [12]:
def mesclar(atual : dict, novo :dict)-> dict:
    return {
        **(atual or {}),
        **(novo or {})
    }

In [13]:
atual = {
    "web": 0.7,
    "python": 0.8
}

novo = {
    "docs": 0.9,
    "web": 1.0
}

##### <span style="color: #4FC1FF;">O operador ** é utilizado para desempacotar os elementos de um dicionário.</span ><br>
Quando fazemos: { **atual, **novo}<br>
É como se estivessemos escrevendo : 
{
    "web": 0.7,
    "python": 0.8,
    "docs": 0.9,
    "web": 1.0
}<br>
Mas existe uma regra importante: quando duas chaves são iguais, o último valor prevalece.<br>
{
    "web": 1.0,
    "python": 0.8,
    "docs": 0.9
}

In [14]:
resultado = mesclar(atual,novo)
print(resultado)

{'web': 1.0, 'python': 0.8, 'docs': 0.9}


#### <span style="color: #4FC1FF;">[3] Maior</span>
O objetivo dessa função é juntar retornar o maior

In [15]:
def maior(atual:float, novo:float)->float:
    return max((atual or 0),(novo or 0))

In [16]:
atual = 0.7
novo = 0.4
print(maior(atual,novo))

0.7


#### <span style="color: #4FC1FF;">[4] Ultimos</span>
O objetivo dessa função é retornar os n ultimos elementos de uma lista de elementos

In [21]:
def ultimos(n: int):
    def reducer(atual:list,novo:list)->list:
        return ((atual or [])+(novo or []))[-n:]
    
    return reducer

Podemos chamar de duas maneiras:

In [23]:
reducer = ultimos(3)
resultado = reducer([1,2,3],[4,5])
print(resultado)

[3, 4, 5]


Ou essa:

In [24]:
print(ultimos(3)([1, 2, 3], [4, 5]))

[3, 4, 5]


A função reducer consegue acessar o valor de n pelo conceito conhecido como closure (fechamento): a função interna consegue acessar a variável n definida na função externa, mesmo depois que a função externa terminou.

#### <span style="color: #4FC1FF;">Onde isso aparece no LangGraph? </span>

O langGraph utiliza um estado compartilhado entre os nodes, o reducer ajuda a organizar esses estado, evitando perdas de informações desnecessarias.

In [25]:
from typing import TypedDict, Annotated

In [26]:
class Estado(TypedDict):
    historico : Annotated[list, lambda a,b : a + b]

In [29]:
concatenar = lambda a,b : a + b
frase_a = ['Olá']
frase_b = ['Tudo bem ?']
print(concatenar(frase_a,frase_b))

['Olá', 'Tudo bem ?']


Agora quando um node retornar uma nova lista para o historico, o estado sabe que deve usar a função lambda para concatenar.

In [49]:
def primeiros(n:int):
    def reducer(atual:list,novo:list)->list:
        return ((atual or [])+(novo or []))[:n]
    return reducer

In [50]:
numeros_a = [1, 2, 3, 4, 5]
numeros_b = [6, 7, 8, 9, 10]

In [51]:
reducer = primeiros(4)
resultado = reducer(numeros_a,numeros_b)
print(resultado)

[1, 2, 3, 4]


#### Desafio

Reducers em conjunto<br>
Agora quero propor algo um pouco mais próximo do que você encontrará trabalhando com agentes.

Imagine que temos um agente que pesquisa documentos e páginas da internet.

Queremos armazenar no estado:
- fontes: lista de fontes consultadas, sem duplicatas.
- pontuacao: maior pontuação de relevância encontrada.
- metadados: dicionário com informações das pesquisas.

Você já possui três reducers que resolvem esses problemas:
- unicos
- maior
- mesclar

In [56]:
from langgraph.graph import START,END, StateGraph

In [64]:
class Estado(TypedDict):
    fontes : Annotated[list[str],unicos]
    pontuacao : Annotated[float,maior]
    metadados : Annotated[dict,mesclar]

In [66]:
def buscaDocs (state: Estado):
    return {'fontes': ['doc1','doc2'],
            'pontuacao': 0.75,
            "metadados": {"origem": "documentos"}}

In [67]:
def buscaWeb(state: Estado):
    return {'fontes': ['doc2','site_1'],
            'pontuacao': 0.92,
            "metadados": {"origem": "web", "total": 3}}

In [68]:
graph = StateGraph(Estado)
graph.add_node('web',buscaWeb)
graph.add_node('docs',buscaDocs)

graph.add_edge(START,'docs')
graph.add_edge('docs','web')
graph.add_edge('web',END)


In [69]:
print(graph.compile().invoke({'fontes': [],'pontuacao': 0,"metadados": {}}))

{'fontes': ['doc1', 'doc2', 'site_1'], 'pontuacao': 0.92, 'metadados': {'origem': 'web', 'total': 3}}
